# Training on medical simplification data
this notebook was continously used (and adapted) for various training setups for t5 modles for training on medical data

In [1]:
#miscellaneous imports
import requests
import os
import datetime
import random
import zipfile
import shutil
import math
import torch
from torch.utils.data import DataLoader
from torch.optim import AdamW # Changed from transformers import AdamW
import datasets as hf_datasets # Alias to avoid conflict with local variables
import transformers as hf_transformers # Alias
from datasets import Dataset, DatasetDict, load_dataset
from transformers import T5Tokenizer, T5TokenizerFast, T5ForConditionalGeneration, set_seed, get_scheduler
from accelerate import Accelerator, notebook_launcher
from tqdm.notebook import tqdm # Or from tqdm import tqdm
import importlib.metadata # For version checking
from transformers import (
    T5Tokenizer,
    T5ForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq,
    set_seed
)
import nltk
import numpy as np

import error: No module named 'triton'


# Download the data (ctrl + 'a' then ctrl + '/')

In [ ]:
# user = "SebaJoe"
# repo = "MultiCochrane"
# branch = "main" 
# path_to_dir = "data/MultiCochrane"

# filenames = [
#     "multiCochrane_all.zip"
# ]

# local_dir = "./data"

# try:
#     os.makedirs(local_dir, exist_ok=True)
#     print(f"Directory '{local_dir}' ensured.")
# except OSError as e:
#     print(f"Error creating directory {local_dir}: {e}")
#     filenames = [] 

# download_count = 0
# error_count = 0
# for filename in filenames:
#     raw_url = f"https://raw.githubusercontent.com/{user}/{repo}/{branch}/{path_to_dir}/{filename}"
#     local_filepath = os.path.join(local_dir, filename)
#     try:
#         response = requests.get(raw_url, stream=True) 
#         response.raise_for_status() 
#         with open(local_filepath, 'wb') as f:
#             for chunk in response.iter_content(chunk_size=8192):
#                 f.write(chunk)
#         download_count += 1
#     except Exception as e:
#         print(f"Error {filename}: {e}")
#         error_count += 1


# def unzip_file(zip_file_path, extract_to_dir):
#     try:
#         os.makedirs(extract_to_dir, exist_ok=True)
#     except OSError as e:
#         print(f"Error{extract_to_dir}: {e}")
#         return 

#     print(f"extracting '{zip_file_path}' to '{extract_to_dir}'...")
#     try:
#         with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
#             zip_ref.extractall(extract_to_dir)
#     except Exception as e:
#         print(f"error {e}")


# path_to_my_zip = local_dir + "/" + filenames[0] 
# destination_folder = local_dir + "/multiCochrane_all"
# unzip_file(path_to_my_zip, destination_folder)
# print("ready freddy")

# Load the unfiltered english data (multiCochrane_all)
put it in transformers/datasets suitable format

In [2]:
import os
import sys
from pathlib import Path
from datasets import load_dataset

_root = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "src" / "paths.py").exists())
sys.path.insert(0, str(_root / "src"))
from paths import MULTICOHRANE_EN_UNFILTERED

base_path = str(MULTICOHRANE_EN_UNFILTERED)
data_files = {
    "train": os.path.join(base_path, "train0_en.csv"),
    "test": os.path.join(base_path, "test0_en.csv"),
    "validation": os.path.join(base_path, "val0_en.csv")
}
multi_cochrane_dataset = load_dataset("csv", data_files=data_files)
print("\nDataset loaded successfully:")
print(multi_cochrane_dataset)




Dataset loaded successfully:
DatasetDict({
    train: Dataset({
        features: ['Unnamed: 0', 'prefix', 'input_text', 'target_text', 'doi'],
        num_rows: 61194
    })
    validation: Dataset({
        features: ['Unnamed: 0', 'prefix', 'input_text', 'target_text', 'doi'],
        num_rows: 83
    })
    test: Dataset({
        features: ['Unnamed: 0', 'prefix', 'input_text', 'target_text', 'doi'],
        num_rows: 395
    })
})


In [68]:
multi_cochrane_dataset['train'][5]

{'Unnamed: 0': 9,
 'prefix': 'simplify: ',
 'input_text': 'One study, the European Organisation for Research and Treatment of Cancer (EORTC) trial, demonstrated a statistically significant overall survival (OS) benefit for RT plus PCV, with a median OS of 3.5 years compared with 2.6 years in the RT alone arm (P value = 0.018).',
 'target_text': 'One study was able to demonstrate a significant survival benefit for the addition of chemotherapy to radiotherapy after surgery, compared with radiotherapy alone.',
 'doi': '10.1002/14651858.CD007104.pub2'}

# Add a prefix to the training data for speficiying simplificatio task

In [3]:
prefix = "simplify: "
def add_prefix(example):
  example['prefix'] = prefix
  return example

for split in ["train", "test", "validation"]:
    updated_train_dataset = multi_cochrane_dataset[split].map(add_prefix)
    multi_cochrane_dataset[split] = updated_train_dataset

print("Updated dataset structure:")
print(multi_cochrane_dataset)
print("\nFirst example of updated train split:")
print(multi_cochrane_dataset["train"][0])

Map:   0%|          | 0/61194 [00:00<?, ? examples/s]

Map:   0%|          | 0/395 [00:00<?, ? examples/s]

Map:   0%|          | 0/83 [00:00<?, ? examples/s]

Updated dataset structure:
DatasetDict({
    train: Dataset({
        features: ['Unnamed: 0', 'prefix', 'input_text', 'target_text', 'doi'],
        num_rows: 61194
    })
    validation: Dataset({
        features: ['Unnamed: 0', 'prefix', 'input_text', 'target_text', 'doi'],
        num_rows: 83
    })
    test: Dataset({
        features: ['Unnamed: 0', 'prefix', 'input_text', 'target_text', 'doi'],
        num_rows: 395
    })
})

First example of updated train split:
{'Unnamed: 0': 0, 'prefix': 'simplify: ', 'input_text': 'Compared to standard care, social skills training may improve the social skills of people with schizophrenia and reduce relapse rates, but at present, the evidence is very limited with data rated as very low quality.', 'target_text': 'However, at the moment evidence is very limited with data only of very low quality available.', 'doi': '10.1002/14651858.CD009006.pub2'}


# Encode using t5 tokenizer

In [4]:
# encode dataset
#based on https://colab.research.google.com/github/NielsRogge/Transformers-Tutorials/blob/master/T5/Fine_tuning_Dutch_T5_base_on_CNN_Daily_Mail_for_summarization_(on_TPU_using_HuggingFace_Accelerate).ipynb#scrollTo=tiLdcTmkg-_o
from transformers import T5Tokenizer
tokenizer = T5Tokenizer.from_pretrained("google-t5/t5-large")

max_input_length = 128
max_target_length = 128

def preprocess_examples(examples):
  input_txt = examples['input_text']
  target_txt = examples['target_text']
  prefix = examples['prefix']
  
  inputs = [prefix + inp for inp, prefix in zip(input_txt, prefix)]
  model_inputs = tokenizer(inputs, max_length=max_input_length, padding="max_length", truncation=True)

  # encode the simplifications
  labels = tokenizer(target_txt, max_length=max_target_length, padding="max_length", truncation=True).input_ids
  labels_with_ignore_index = []
  for labels_example in labels:
    labels_example = [label if label != 0 else -100 for label in labels_example]
    labels_with_ignore_index.append(labels_example)
  
  model_inputs["labels"] = labels_with_ignore_index

  return model_inputs

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


In [5]:
train_ds = multi_cochrane_dataset['train']
val_ds = multi_cochrane_dataset['validation']
test_ds = multi_cochrane_dataset['test']
encoded_train_ds = train_ds.map(preprocess_examples, batched=True, remove_columns=train_ds.column_names)
encoded_val_ds = val_ds.map(preprocess_examples, batched=True, remove_columns=val_ds.column_names)
encoded_test_ds = test_ds.map(preprocess_examples, batched=True, remove_columns=test_ds.column_names)

Map:   0%|          | 0/61194 [00:00<?, ? examples/s]

Map:   0%|          | 0/83 [00:00<?, ? examples/s]

Map:   0%|          | 0/395 [00:00<?, ? examples/s]

In [6]:
encoded_train_ds

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 61194
})

In [7]:
encoded_train_ds['labels'][0]

[611,
 6,
 44,
 8,
 798,
 2084,
 19,
 182,
 1643,
 28,
 331,
 163,
 13,
 182,
 731,
 463,
 347,
 5,
 1,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100]

In [8]:
encoded_train_ds['input_ids'][0]

[18356,
 10,
 3,
 25236,
 12,
 1068,
 124,
 6,
 569,
 1098,
 761,
 164,
 1172,
 8,
 569,
 1098,
 13,
 151,
 28,
 31926,
 11,
 1428,
 3,
 60,
 16543,
 1917,
 6,
 68,
 44,
 915,
 6,
 8,
 2084,
 19,
 182,
 1643,
 28,
 331,
 3,
 4094,
 38,
 182,
 731,
 463,
 5,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0]

# Set up some default hyper parameters, initialize modle, data_collator and adamW optimzier

In [9]:
hyperparameters = {
    "learning_rate": 1e-5, 
    "num_epochs": 25, 
    "train_batch_size": 16, 
    #"gradient_accumulation_steps": 4,
    "eval_batch_size": 4, 
    "seed": 42,
    "output_dir": "./content_simplifier_t5_base_ROUGE_unfiltered_epoch_25/",
    "mixed_precision": "fp16",
}

In [11]:
#set up model and optimizer
import evaluate
model_location = r"D:\trained_models\optuna_sweep_checkpoints_t5_small_pretrained_wikilarge\run-4\checkpoint-960" 

set_seed(hyperparameters["seed"])
model = T5ForConditionalGeneration.from_pretrained(model_location)
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)
optimizer = AdamW(model.parameters(), lr=hyperparameters["learning_rate"])
#sari_metric = evaluate.load("sari")
nltk.download("punkt", quiet=True)
metric = evaluate.load("rouge")

# Sari metric eval
metric used for evalutation of simplification task

In [ ]:
sari_metric = evaluate.load("sari")
def compute_metrics_sari(eval_preds):
    predictions, labels, inputs = eval_preds.predictions, eval_preds.label_ids, eval_preds.inputs
    predictions = np.clip(predictions, 0, tokenizer.vocab_size - 1)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    inputs = np.where(inputs != -100, inputs, tokenizer.pad_token_id)
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    decoded_inputs = tokenizer.batch_decode(inputs, skip_special_tokens=True)
    all_refs = [[lbl] for lbl in decoded_labels]
    sari_results = sari_metric.compute(
        sources=decoded_inputs,
        predictions=decoded_preds,
        references=all_refs
    )
    return {"sari": sari_results["sari"]}


# Rouge metric eval
(old, but kept for archival/experimental purposese)

In [13]:
# #https://huggingface.co/docs/evaluate/en/transformers_integrations
# import nltk
# nltk.download('punkt_tab')
# def compute_metrics_rouge(eval_preds):
#     preds, labels = eval_preds

#     # Zorg ervoor dat preds en labels geldige token-ID's bevatten
#     preds = np.clip(preds, 0, tokenizer.vocab_size - 1)  # Beperk tot geldige range
#     labels = np.where(labels == -100, tokenizer.pad_token_id, labels)  # Vervang -100 door pad_token_id

#     # Decode de voorspellingen en labels
#     decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
#     decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

#     # Voeg newlines toe voor rougeLSum
#     decoded_preds = ["\n".join(nltk.sent_tokenize(pred.strip())) for pred in decoded_preds]
#     decoded_labels = ["\n".join(nltk.sent_tokenize(label.strip())) for label in decoded_labels]

#     # Bereken de ROUGE-score
#     result = metric.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)
#     return result

# run below for training for speficic hyperparameters, but skip for sweep

In [14]:
# training_args = Seq2SeqTrainingArguments(
#     output_dir=hyperparameters["output_dir"],
#     overwrite_output_dir=True,
#     evaluation_strategy="epoch",
#     save_strategy="epoch",
#     num_train_epochs=hyperparameters["num_epochs"],
#     learning_rate=hyperparameters["learning_rate"],
#     lr_scheduler_type="linear",
#     warmup_steps=500,
#     per_device_train_batch_size=16, 
#     per_device_eval_batch_size=16,
#     weight_decay=0.01,
#     predict_with_generate=True,
#     include_inputs_for_metrics=True,
#     gradient_accumulation_steps=2,
#     generation_max_length=128,
#     load_best_model_at_end=True,
#     optim="adamw_torch",
#     fp16=True,
#     label_smoothing_factor=0.1,
#     metric_for_best_model="sari",
#     greater_is_better=True,
#     report_to = "none")

In [15]:
# trainer = Seq2SeqTrainer(
#     model=model,
#     args=training_args,
#     train_dataset=encoded_train_ds,
#     eval_dataset=encoded_val_ds,
#     data_collator=data_collator,
#     tokenizer=tokenizer,
#     compute_metrics=compute_metrics_sari
# )

In [16]:
# train_result = trainer.train()
# trainer.save_model(training_args.output_dir)



In [17]:
# import torch
# i=1
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# prompt = f"Simplify text: {multi_cochrane_dataset["train"][i]["input_text"]}"
# inputs = tokenizer(prompt, return_tensors="pt").to(device)
# generated_ids = trainer.model.generate(**inputs, max_length=max_target_length, do_sample=True, top_p=0.95, temperature=0.1)
# summary = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
# print(f"original: {multi_cochrane_dataset["train"][i]["input_text"]}") 
# print("Simplified:", summary)

# hyperparameter sweep using optuna

In [ ]:
hyperparameters

Often start runs form here, so for safety, reiinit everything

In [ ]:
import evaluate
set_seed(hyperparameters["seed"])
model_location = r"D:\trained_models\optuna_sweep_checkpoints_t5_small_pretrained_wikilarge\run-4\checkpoint-960" 
model = T5ForConditionalGeneration.from_pretrained(model_location)
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)
optimizer = AdamW(model.parameters(), lr=hyperparameters["learning_rate"])
sari_metric = evaluate.load("sari")
# nltk.download("punkt", quiet=True)
# metric = evaluate.load("rouge")

In [ ]:
#general hyperparameters, basically set according to the GPU performance, model size and data size
hyperparameters = {
    "train_batch_size": 64,
    "eval_batch_size": 64, 
    "seed": 42,
    "mixed_precision": "fp16", 
    "gradient_accumulation_steps": 2,
}
set_seed(hyperparameters["seed"])


In [15]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./optuna_sweep_checkpoints_t5_small_pretrained_wikilarge", 
    overwrite_output_dir=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    per_device_train_batch_size=hyperparameters["train_batch_size"],
    per_device_eval_batch_size=hyperparameters["eval_batch_size"],
    predict_with_generate=True,
    include_for_metrics=["inputs"],
    gradient_accumulation_steps=hyperparameters["gradient_accumulation_steps"],
    generation_max_length=128,
    load_best_model_at_end=True, 
    optim="adamw_torch",
    fp16=True,
    metric_for_best_model="sari",
    greater_is_better=True,      
    report_to="none",              
    seed=hyperparameters["seed"],  
    save_total_limit=1            
)

In [16]:
trainer = Seq2SeqTrainer(
    args=training_args,
    model_init=model_init, 
    train_dataset=encoded_train_ds,
    eval_dataset=encoded_val_ds,
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics_sari
)

In [17]:
N_TRIALS = 20
print(f"Starting hyperparameter search with {N_TRIALS} trials...")

study = optuna.create_study(
    direction="maximize",
    study_name="t5-large-sari-sweep", 
)

[I 2025-04-10 15:26:53,360] A new study created in memory with name: t5-large-sari-sweep


Starting hyperparameter search with 20 trials...


In [18]:
best_run = trainer.hyperparameter_search(
    direction="maximize",               # We want to maximize SARI
    backend="optuna",                   # Use Optuna backend
    hp_space=hp_space,                  # Function defining the search space (Trainer will pass trial to it)
    n_trials=N_TRIALS,                  # Number of trials for the Trainer to run
    compute_objective=compute_objective, # Function to get the objective value from metrics
    study_name="t5-sari-sweep",         # Optional: Name the study managed internally by Trainer
    load_if_exists=True,                # Optional: Resume the study if storage exists

)

[I 2025-04-10 15:26:55,864] A new study created in memory with name: t5-sari-sweep
Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Epoch,Training Loss,Validation Loss


[W 2025-04-10 15:27:20,619] Trial 0 failed with parameters: {'learning_rate': 0.0002862961213247954, 'num_train_epochs': 4, 'weight_decay': 0.04322940045181294, 'label_smoothing_factor': 0.16314435776067657, 'warmup_steps': 723, 'lr_scheduler_type': 'linear'} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "c:\Users\Ruben\GPUcodig\AIenv\Lib\site-packages\optuna\study\_optimize.py", line 197, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "c:\Users\Ruben\GPUcodig\AIenv\Lib\site-packages\transformers\integrations\integration_utils.py", line 254, in _objective
    trainer.train(resume_from_checkpoint=checkpoint, trial=trial)
  File "c:\Users\Ruben\GPUcodig\AIenv\Lib\site-packages\transformers\trainer.py", line 2245, in train
    return inner_training_loop(
           ^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Ruben\GPUcodig\AIenv\Lib\site-packages\transformers\trainer.py", line 2556, in _inner_training_loo

KeyboardInterrupt: 

In [ ]:
print("Hyperparameter search finished.")

print(f"Best run found: Run ID {best_run.run_id}") 
print(f"Best SARI score: {best_run.objective}")
print("Best hyperparameters:")
print(best_run.hyperparameters) 

Hyperparameter search finished.
Best run found: Run ID 17
Best SARI score: 42.61698282898968
Best hyperparameters:
{'learning_rate': 0.00013795652077000847, 'num_train_epochs': 5, 'weight_decay': 0.039744046312776046, 'label_smoothing_factor': 0.16026143407163396, 'warmup_steps': 774, 'lr_scheduler_type': 'constant_with_warmup'}


In [ ]:
#set it manually to the best hyperparameters (had difficulties loading it from checkpoint)
best_hyperparameters = {
    "learning_rate": 0.0003087545189866897,
    "num_train_epochs": 15, 
    "lr_scheduler_type": "constant_with_warmup",
    "warmup_steps": 789, 
    "weight_decay": 0.057960328420691556,
    "label_smoothing_factor": 0.1548218707925769,
}

In [ ]:
#train the final model with the best hyperparameters
print("\nTraining final model with best hyperparameters...")
if best_hyperparameters is None:
     raise RuntimeError("Hyperparameter search did not return best hyperparameters.")
print(f"Applying best hyperparameters: {best_hyperparameters}")
final_output_dir = "./final_t5_small_opt_sweep_pretrained_wiki_run4_check960_RUN2/" 
final_training_args = Seq2SeqTrainingArguments(
    output_dir=final_output_dir,
    overwrite_output_dir=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    per_device_train_batch_size=hyperparameters["train_batch_size"],
    per_device_eval_batch_size=hyperparameters["eval_batch_size"],
    predict_with_generate=True,
    include_for_metrics=["inputs"],
    gradient_accumulation_steps=2,
    generation_max_length=128,
    load_best_model_at_end=True,
    optim="adamw_torch",
    fp16=(hyperparameters["mixed_precision"] == "fp16"),
    metric_for_best_model="sari",
    greater_is_better=True,
    report_to="tensorboard",
    seed=hyperparameters["seed"],
    save_total_limit=2, 
    logging_strategy="steps", 
    logging_steps=100,
    num_train_epochs=30,#best_hyperparameters["num_train_epochs"],
    learning_rate=best_hyperparameters["learning_rate"],
    lr_scheduler_type=best_hyperparameters["lr_scheduler_type"],
    warmup_steps=best_hyperparameters["warmup_steps"],
    weight_decay=best_hyperparameters["weight_decay"],
    label_smoothing_factor=best_hyperparameters["label_smoothing_factor"],
)


final_trainer = Seq2SeqTrainer(
    model=model_init(None), 
    args=final_training_args,
    train_dataset=encoded_train_ds,
    eval_dataset=encoded_val_ds,
    data_collator=data_collator,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics_sari
)


print("Starting final training run...")
train_result = final_trainer.train()
print("Final training complete.")

final_trainer.save_model(final_output_dir)
final_trainer.save_state() # Save optimizer state etc.
tokenizer.save_pretrained(final_output_dir)
print(f"Final best model saved to {final_output_dir}")



Training final model with best hyperparameters...
Applying best hyperparameters: {'learning_rate': 0.0003087545189866897, 'num_train_epochs': 15, 'lr_scheduler_type': 'constant_with_warmup', 'warmup_steps': 789, 'weight_decay': 0.057960328420691556, 'label_smoothing_factor': 0.1548218707925769}


C:\Users\Ruben\AppData\Local\Temp\ipykernel_41360\1966569801.py:50: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  final_trainer = Seq2SeqTrainer(


Starting final training run...


Epoch,Training Loss,Validation Loss,Sari
1,3.474700,3.767340,45.249081
2,3.470400,3.744565,45.627198
3,3.442900,3.749343,46.542330
4,3.407300,3.768689,44.961901
5,3.417100,3.767105,46.664648
6,3.364200,3.759534,45.223046
7,3.378700,3.760846,46.339167
8,3.345000,3.765703,45.308867
9,3.323900,3.778713,44.976708
10,3.300200,3.789542,44.861903


There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


Final training complete.
Final best model saved to ./final_t5_small_opt_sweep_pretrained_wiki_run4_check960_RUN2/


In [61]:
multi_cochrane_dataset['train'][0]

{'Unnamed: 0': 0,
 'prefix': 'simplify: ',
 'input_text': 'Compared to standard care, social skills training may improve the social skills of people with schizophrenia and reduce relapse rates, but at present, the evidence is very limited with data rated as very low quality.',
 'target_text': 'However, at the moment evidence is very limited with data only of very low quality available.',
 'doi': '10.1002/14651858.CD009006.pub2'}

In [88]:
input_dir = final_output_dir 

In [ ]:
#final_output_dir = r"C:\Users\Ruben\GPUcodig\LM\medical_project_simplification\optuna_sweep_checkpoints_t5_large\run-0\checkpoint-7650"
device = torch.cuda.current_device() if torch.cuda.is_available() else "cpu"
model = T5ForConditionalGeneration.from_pretrained(final_output_dir).to(device)
tokenizer = T5Tokenizer.from_pretrained("google-t5/t5-large")

for i in range(1000):
    prompt = f"simplify: {multi_cochrane_dataset['train'][i]['input_text']}"
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    generated_ids = model.generate(**inputs,
                                    max_length=128, 
                                    temperature=0.7,
                                    num_beams=50)#,
                                    #length_penalty=1.0,
                                    #no_repeat_ngram_size=5)
    summary = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
    print(f"Original: {multi_cochrane_dataset['train'][i]['input_text']}")
    print(f"Simplified: {summary}\n\n")

Original: Compared to standard care, social skills training may improve the social skills of people with schizophrenia and reduce relapse rates, but at present, the evidence is very limited with data rated as very low quality.
Simplified: Compared to standard care, social skills training may improve the social skills of people with schizophrenia and reduce relapse rates.


Original: Four out of 52 studies, including 128 CCS, assessed the prevalence of hypomagnesaemia, which ranged between 13.2% and 28.6%.
Simplified: Four out of 52 studies, including 128 CCS, assessed the prevalence of hypomagnesaemia, which ranged between 13.2% and 28.6%.


Original: Seven RCTs including 960 participants were identified.
Simplified: The review authors identified seven randomised controlled trials (RCTs) including 960 participants that compared L-ornithine L-ornithine L-ornithine L-ornithine L-ornithine L-ornithine L-ornithine L-ornithine L-ornithine L-ornithine L-ornithine L-ornithine L-ornithine L-as